# RunnableWithMessageHistory 示例代码解析
在这一部分，我们将逐步分析给定代码的每个部分，解释其功能与逻辑。此代码演示了如何使用 `RunnableWithMessageHistory` 类结合 `chain` 进行消息历史管理。

### `get_by_session_id` 函数
此函数用于根据会话ID（`session_id`）从`store`中获取消息历史。如果历史记录不存在，它将初始化一个新的 `InMemoryChatMessageHistory` 对象并将其存储在 `store` 中。
这一部分确保了每个会话都拥有独立的历史记录，避免不同会话之间的消息干扰。

关键功能：
- 检查 `session_id` 是否已在 `store` 中存在历史记录。
- 如果不存在，则初始化一个新的 `InMemoryChatMessageHistory` 并返回。
- 返回与 `session_id` 相关联的历史记录。

In [1]:
from langchain_core.chat_history import (
    BaseChatMessageHistory,  # 基础聊天消息历史类
    InMemoryChatMessageHistory,  # 内存中的聊天消息历史类
)
import os
store = {}

def get_by_session_id(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


### `ChatPromptTemplate` 和 `ChatOpenAI` 的组合
在这一部分，我们定义了一个 `ChatPromptTemplate` 来构建消息格式，该格式包括：
- 系统消息（`system`）：用来设置助手的能力。
- 历史消息（`history`）：由 `MessagesPlaceholder` 占位符表示，用于在对话中插入历史消息。
- 用户问题（`human`）：这是用户当前提出的问题。

然后，我们将该模板与 `ChatOpenAI` 模型结合，`ChatOpenAI` 模型是用于生成回答的核心组件，结合提供的 `model` 和 `openai_api_key`。

这段代码定义了一个 `chain`，它将从 `prompt` 到 `ChatOpenAI` 进行信息流转。

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

prompt = ChatPromptTemplate.from_messages([
    ("system", "You're an assistant who's good at {ability}"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}"),
])

chain = prompt | ChatOpenAI(model="gpt-4o-mini", openai_api_key=os.environ.get("aihubmix_key"),
                            openai_api_base="https://api.aihubmix.com/v1")  # 使用的模型名称


### `RunnableWithMessageHistory` 的定义与初始化
在这部分代码中，我们将 `chain` 和 `get_by_session_id` 函数传入 `RunnableWithMessageHistory` 类，以便实现消息历史的管理。

关键功能：
- `input_messages_key="question"`：指定从输入中提取 `question` 作为问题输入。
- `history_messages_key="history"`：指定历史消息应存储在 `history` 中，并传递给模型。

`RunnableWithMessageHistory` 在执行时会自动管理并传递会话历史记录，这使得 `chain` 可以在多次调用之间保持对话上下文。


In [3]:
chain_with_history = RunnableWithMessageHistory(
    chain,
    # 使用之前定义的 get_by_session_id 函数
    get_by_session_id,
    input_messages_key="question",
    history_messages_key="history",
)


### 第一次调用 `chain_with_history.invoke`
在此调用中，我们传入了能力（`ability`）和问题（`question`）。`chain_with_history.invoke` 会将这些输入数据与历史消息一起传递给 `chain`，并生成模型的响应。

调用的结果会被存储在 `store` 中，以便下一次调用使用。

这时，`store` 会存储会话 ID `foo` 下的历史消息，包括用户问题和模型回答。

In [4]:
content = chain_with_history.invoke(  
    {"ability": "math", "question": "What does cosine mean?"},
    config={"configurable": {"session_id": "foo"}})
from pprint import pprint
pprint(vars(content))

{'additional_kwargs': {'refusal': None},
 'content': 'Cosine is a mathematical function that relates to the angles and '
            'sides of a right triangle. Specifically, for a given angle \\( '
            '\\theta \\) in a right triangle, the cosine of that angle is '
            'defined as the ratio of the length of the adjacent side to the '
            'length of the hypotenuse. \n'
            '\n'
            'Mathematically, this is expressed as:\n'
            '\n'
            '\\[\n'
            '\\cos(\\theta) = \\frac{\\text{adjacent}}{\\text{hypotenuse}}\n'
            '\\]\n'
            '\n'
            'Cosine is also a fundamental function in trigonometry and is used '
            'in various branches of mathematics, physics, and engineering. It '
            'can be extended to the unit circle, where for an angle \\( '
            '\\theta \\), the cosine value represents the x-coordinate of the '
            'point on the unit circle corresponding to that angle.

In [10]:
pprint(store['foo'])

InMemoryChatMessageHistory(messages=[HumanMessage(content='What does cosine mean?'), AIMessage(content='Cosine is a mathematical function that relates to the angles and sides of a right triangle. Specifically, for a given angle \\( \\theta \\) in a right triangle, the cosine of that angle is defined as the ratio of the length of the adjacent side to the length of the hypotenuse. \n\nMathematically, this is expressed as:\n\n\\[\n\\cos(\\theta) = \\frac{\\text{adjacent}}{\\text{hypotenuse}}\n\\]\n\nCosine is also a fundamental function in trigonometry and is used in various branches of mathematics, physics, and engineering. It can be extended to the unit circle, where for an angle \\( \\theta \\), the cosine value represents the x-coordinate of the point on the unit circle corresponding to that angle.\n\nThe cosine function has important properties, including:\n\n- It is periodic with a period of \\( 2\\pi \\) radians (or 360 degrees).\n- It is an even function, meaning \\( \\cos(-\\thet

### 第二次调用 `chain_with_history.invoke`
在第二次调用时，问题变更为“What's its inverse”，此时历史消息已经包含了之前的对话内容。

`RunnableWithMessageHistory` 会再次调用 `get_by_session_id`，获取会话ID为 `foo` 的历史记录，然后将当前问题与历史消息一起传递给 `chain`。

模型根据历史消息和当前的问题生成新的回答，并将其存储在 `store` 中，以便下一次调用使用。

In [11]:
content = chain_with_history.invoke(  
    {"ability": "math", "question": "What's its inverse"},
    config={"configurable": {"session_id": "foo"}})
pprint(vars(content))

{'additional_kwargs': {'refusal': None},
 'content': 'The inverse of the cosine function is called the **arccosine**, '
            'denoted as \\( \\cos^{-1}(x) \\) or \\( \\text{arccos}(x) \\). '
            'The arccosine function takes a value from the range of the cosine '
            'function (which is \\([-1, 1]\\)) and returns the angle whose '
            'cosine is that value.\n'
            '\n'
            'Mathematically, if \\( y = \\cos(x) \\), then:\n'
            '\n'
            '\\[\n'
            'x = \\cos^{-1}(y) \\quad \\text{or} \\quad x = \\text{arccos}(y)\n'
            '\\]\n'
            '\n'
            'The output of the arccosine function is typically restricted to '
            'the range \\([0, \\pi]\\) radians (or \\([0^\\circ, '
            '180^\\circ]\\)) to ensure that it returns a unique angle for each '
            'input in the range of the cosine function.\n'
            '\n'
            'For example:\n'
            '- \\( \\cos^{-1}(1) = 0 \\

In [14]:
pprint(store['foo'])

InMemoryChatMessageHistory(messages=[HumanMessage(content='What does cosine mean?'), AIMessage(content='Cosine is a mathematical function that relates to the angles and sides of a right triangle. Specifically, for a given angle \\( \\theta \\) in a right triangle, the cosine of that angle is defined as the ratio of the length of the adjacent side to the length of the hypotenuse. \n\nMathematically, this is expressed as:\n\n\\[\n\\cos(\\theta) = \\frac{\\text{adjacent}}{\\text{hypotenuse}}\n\\]\n\nCosine is also a fundamental function in trigonometry and is used in various branches of mathematics, physics, and engineering. It can be extended to the unit circle, where for an angle \\( \\theta \\), the cosine value represents the x-coordinate of the point on the unit circle corresponding to that angle.\n\nThe cosine function has important properties, including:\n\n- It is periodic with a period of \\( 2\\pi \\) radians (or 360 degrees).\n- It is an even function, meaning \\( \\cos(-\\thet

# `RunnableWithMessageHistory` 与 `chain` 的信息流动解析
在这一部分，我们将深入分析 `RunnableWithMessageHistory` 与 `chain` 之间如何流动信息，确保多轮对话的上下文能够得以保存和传递。

## 问题输入：信息流的起点
在本示例中，我们的起点是一个问题：“What does cosine mean?”，该问题将作为输入传递给模型。为了保持对话的上下文，我们还需要依赖消息历史来构建对话的连续性。

```python
result_1 = chain_with_history.invoke(
    {"ability": "math", "question": "What does cosine mean?"}, 
    config={"configurable": {"session_id": "foo"}}
)
print(result_1)
```


### 信息流分析 - 第一次调用
1. **输入数据**：
- `{ "ability": "math", "question": "What does cosine mean?" }`：这是传入`invoke`方法的数据，包含了两个信息：
    - `ability`：表明模型要展示的能力，这里是数学（`math`）。
    - `question`：实际的问题，要求模型解释余弦（cosine）的含义。

2. **消息历史的作用**：
- `RunnableWithMessageHistory` 在这个调用时，会去调用我们之前定义的 `get_by_session_id` 函数，查找会话ID为`foo`的历史记录。
- 由于这是第一次调用，`get_by_session_id`将返回一个新的`InMemoryChatMessageHistory`对象，该对象暂时不包含任何历史消息。

3. **历史消息填充**：
- `RunnableWithMessageHistory` 会将当前问题（"What does cosine mean?"）和历史消息（为空）一同传递给`chain`（即`ChatOpenAI`模型）。
- 具体来说，`RunnableWithMessageHistory` 使用`input_messages_key="question"`和`history_messages_key="history"`将数据传递给模型，其中历史消息为空，模型仅根据当前的问题生成回答。

4. **模型生成回答**：
- `chain` 会使用`ChatOpenAI`模型，基于提供的模板和当前的输入生成一个回答。
- 模型生成的回答会被返回，并且这个回答会成为未来调用中的历史消息之一。

### 输出结果
在第一次调用后，我们可以查看返回结果并打印存储的消息历史。
```python
print(store),
```
此时，`store`中存储了会话ID为`foo`的历史记录，包括输入问题和模型的回答。


## 第二次调用：信息流的延续

在第二次调用时，问题变为“What's its inverse”。此时，历史消息已经不再是空的，因为它包含了第一次的问答对。

```python
result_2 = chain_with_history.invoke(
    {"ability": "math", "question": "What's its inverse"},
    config={"configurable": {"session_id": "foo"}}
)
print(result_2)
```


### 信息流分析 - 第二次调用

1. **输入数据**：
- `{ "ability": "math", "question": "What's its inverse" }`：这是第二次调用时传入的问题数据，询问的是“余弦的反函数是什么”。

2. **历史消息的传递**：
- 这次调用时，`RunnableWithMessageHistory` 会根据会话ID `foo` 获取历史消息。在`get_by_session_id`的作用下，它会返回一个`InMemoryChatMessageHistory`对象，这个对象现在包含了第一次调用中的问题和回答。
- 此时，历史消息会作为`history`被传递给模型，`question`会作为当前输入传递。

3. **信息流的传递**：
- `RunnableWithMessageHistory` 会将历史消息和当前的问题一起打包，传递给`chain`。这意味着，模型不仅需要根据当前问题生成答案，还需要根据历史消息来理解问题的上下文。
- 由于历史消息中已经包含了第一次的对话内容（"What does cosine mean?"以及它的回答），模型能够基于这个上下文来给出关于“cosine反函数”的更准确的回答。

4. **模型生成回答**：
- 模型在接收到完整的上下文信息后，生成并返回关于“cosine的反函数”的解释。这个答案会成为新的历史记录的一部分，供下次调用时使用。

### 输出结果

在第二次调用后，我们可以再次打印`store`，查看历史记录是否已更新。

```python
print(store)
```

此时，`store`中已经包含了完整的历史记录，包括：
- 第一次的提问“What does cosine mean?”及其回答。
- 第二次的提问“What's its inverse”及其回答。



## 关键概念：信息流如何在 `RunnableWithMessageHistory` 中流通

- **输入传递**：每次调用`invoke`方法时，用户传入的问题通过`input_messages_key`传递给模型，模型根据这些信息生成回答。

- **历史记录传递**：每次调用时，`RunnableWithMessageHistory` 会根据会话ID获取历史消息，并通过`history_messages_key`传递给模型。这样，模型不仅处理当前的问题，还考虑到历史上下文。

- **链式结构**：`chain`是一个由多个组件构成的处理链，其中`ChatPromptTemplate`生成聊天提示，`ChatOpenAI`生成回答，而`RunnableWithMessageHistory`则负责在每次调用时管理并传递历史消息。这个链式结构使得信息能够在各个组件之间流动，从而确保对话的连续性。

- **会话隔离**：通过`session_id`，`RunnableWithMessageHistory`能够确保每个会话的历史消息不会混淆。每个会话都有独立的历史记录，这保证了不同用户或不同对话之间的消息不会相互干扰。
